In [1]:
!pip install paho-mqtt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 2.3 MB/s eta 0:00:00


In [2]:
import paho.mqtt.client as mqtt
import json, time, random, math

BROKER = "broker.hivemq.com"
PORT   = 1883
TOPIC_DATA  = "bridge/sensor/data"
TOPIC_ALERT = "bridge/sensor/alert"
DEVICE_ID   = "bridge-node-01"

client = mqtt.Client(client_id="colab-simulator")
client.connect(BROKER, PORT, 60)
client.loop_start()

print("Simulator running... (stop cell to halt)\n")

t = 0
try:
    while True:
        t += 0.1

        # Simulate normal vibration (8 Hz dominant, like truck traffic)
        ax = 0.5 * math.sin(2 * math.pi * 8 * t) + random.gauss(0, 0.05)
        ay = 0.3 * math.sin(2 * math.pi * 8 * t + 0.5) + random.gauss(0, 0.05)
        az = 9.8 + random.gauss(0, 0.1)   # gravity

        # Every ~30 seconds, inject an anomaly burst
        is_anomaly = (int(t) % 30 == 0)
        if is_anomaly:
            ax += 1.2 * math.sin(2 * math.pi * 3 * t)  # low-freq resonance shift
            ax += 0.6 * math.sin(2 * math.pi * 9 * t)  # 3rd harmonic

        payload = json.dumps({
            "device": DEVICE_ID,
            "timestamp": round(t, 2),
            "ax": round(ax, 4),
            "ay": round(ay, 4),
            "az": round(az, 4),
            "temp_c": round(28.5 + random.gauss(0, 0.2), 2)
        })
        client.publish(TOPIC_DATA, payload)
        print(f"[{round(t,1)}s] ax={round(ax,3):>7}  ay={round(ay,3):>7}  az={round(az,3):>7}  {'⚠ ANOMALY' if is_anomaly else ''}")

        # Publish alert separately when anomaly detected
        if is_anomaly:
            score = round(random.uniform(0.7, 0.95), 3)
            alert = json.dumps({
                "device": DEVICE_ID,
                "alert": "STRUCTURAL_ANOMALY_DETECTED",
                "score": score,
                "severity": "HIGH" if score > 0.85 else "MEDIUM",
                "message": "Unusual vibration pattern detected."
            })
            client.publish(TOPIC_ALERT, alert, retain=True)
            print(f"  >>> ALERT published (score={score})")

        time.sleep(0.5)

except KeyboardInterrupt:
    client.loop_stop()
    print("Simulator stopped.")

/tmp/ipykernel_1948/3785585945.py:10: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id="colab-simulator")


Simulator running... (stop cell to halt)

[0.1s] ax=  0.291  ay= -0.195  az=  9.794  ⚠ ANOMALY
  >>> ALERT published (score=0.819)
[0.2s] ax= -1.567  ay= -0.295  az=  9.743  ⚠ ANOMALY
  >>> ALERT published (score=0.71)
[0.3s] ax= -0.978  ay=   0.03  az=  9.886  ⚠ ANOMALY
  >>> ALERT published (score=0.909)
[0.4s] ax=  1.244  ay=  0.262  az=  9.774  ⚠ ANOMALY
  >>> ALERT published (score=0.878)
[0.5s] ax= -0.006  ay=  0.122  az=  9.716  ⚠ ANOMALY
  >>> ALERT published (score=0.821)
[0.6s] ax=  -1.27  ay= -0.236  az=  9.831  ⚠ ANOMALY
  >>> ALERT published (score=0.906)
[0.7s] ax=  1.023  ay= -0.303  az=  9.762  ⚠ ANOMALY
  >>> ALERT published (score=0.754)
[0.8s] ax=  1.569  ay=  0.062  az=   9.77  ⚠ ANOMALY
  >>> ALERT published (score=0.886)
[0.9s] ax= -0.339  ay=  0.305  az=  9.845  ⚠ ANOMALY
  >>> ALERT published (score=0.784)
[1.0s] ax= -0.011  ay=  0.151  az=  9.647  ⚠ ANOMALY
  >>> ALERT published (score=0.738)
[1.1s] ax= -0.459  ay= -0.197  az=  9.689  
[1.2s] ax= -0.277  ay= -0